# Choice Complexity in LLMs — Lean Demo Colab

This notebook demonstrates the current first-paper direction of the project:

- compute a simple, interpretable Choice Complexity Index (CCI) for a set of candidate options,
- compare fixed-size selection strategies,
- and inspect whether complexity-aware selection can reduce redundancy and ambiguity without simply showing fewer options.

This notebook is intentionally narrow. It does **not** present the full long-term research agenda, and it does **not** treat the current CCI formulation as final.


## What this notebook demonstrates

The first paper is centered on a set-level control question:

> given a candidate option set, can we choose a final set that is easier for a user to choose from while preserving usefulness?

To keep the demo simple and reproducible, this notebook uses a lightweight synthetic option generator and a simple utility model. The focus is on the structure of the comparison, not on claiming final empirical results.


## Setup


In [ ]:
# Clone the repository
!git clone https://github.com/soroushbagheri/choice-complexity-llm.git
%cd choice-complexity-llm

# Lightweight dependencies only
!pip install -q numpy pandas scikit-learn matplotlib ipywidgets

import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, Markdown


## Configuration

The final shown set size is kept fixed across strategies. This is important: otherwise a method can appear better simply because it shows fewer options.


In [ ]:
SEED = 42
N_PROBLEMS = 80
CANDIDATE_SET_SIZE = 12
FINAL_SET_SIZE = 4
EMBED_DIM = 12

random.seed(SEED)
np.random.seed(SEED)

print({
    "seed": SEED,
    "n_problems": N_PROBLEMS,
    "candidate_set_size": CANDIDATE_SET_SIZE,
    "final_set_size": FINAL_SET_SIZE,
    "embed_dim": EMBED_DIM,
})


## Helper functions

We use a simple current-stage CCI formulation:

\[
CCI(S) = w_1 \cdot N(S) + w_2 \cdot H_u(S) + w_3 \cdot R(S) + w_4 \cdot A(S)
\]

Where:

- `N(S)`: normalized set size
- `H_u(S)`: entropy over option utilities
- `R(S)`: semantic redundancy
- `A(S)`: top-option ambiguity


In [ ]:
def softmax(x):
    x = np.asarray(x, dtype=float)
    z = np.exp(x - np.max(x))
    return z / z.sum()

def normalized_entropy(values):
    p = softmax(values)
    h = -np.sum(p * np.log(p + 1e-12))
    return h / math.log(len(values))

def redundancy_score(embeddings):
    sims = cosine_similarity(embeddings)
    n = sims.shape[0]
    vals = []
    for i in range(n):
        for j in range(i + 1, n):
            vals.append(sims[i, j])
    if not vals:
        return 0.0
    # map cosine [-1,1] to [0,1]
    vals = [(v + 1) / 2 for v in vals]
    return float(np.mean(vals))

def top_option_ambiguity(utilities):
    ordered = sorted(utilities, reverse=True)
    if len(ordered) < 2:
        return 0.0
    margin = ordered[0] - ordered[1]
    # smaller margin => higher ambiguity
    return float(1 / (1 + math.exp(3 * margin)))

def compute_cci(utilities, embeddings, weights=None):
    if weights is None:
        weights = {"size":0.15, "entropy":0.35, "redundancy":0.30, "ambiguity":0.20}
    n = len(utilities)
    size_term = n / CANDIDATE_SET_SIZE
    entropy_term = normalized_entropy(utilities)
    redundancy_term = redundancy_score(embeddings)
    ambiguity_term = top_option_ambiguity(utilities)
    cci = (
        weights["size"] * size_term
        + weights["entropy"] * entropy_term
        + weights["redundancy"] * redundancy_term
        + weights["ambiguity"] * ambiguity_term
    )
    return {
        "cci": float(cci),
        "size_term": float(size_term),
        "entropy_term": float(entropy_term),
        "redundancy_term": float(redundancy_term),
        "ambiguity_term": float(ambiguity_term),
    }

def utility_distribution_gap(selected_utilities):
    ordered = sorted(selected_utilities, reverse=True)
    return ordered[0] - np.mean(ordered[1:]) if len(ordered) > 1 else ordered[0]


## Synthetic option-set generator

This is only a demonstration scaffold. It generates option sets with:

- latent semantic clusters,
- overlapping candidates,
- a hidden utility score,
- and mild noise so strategies behave differently.


In [ ]:
def make_problem(problem_id, n_options=CANDIDATE_SET_SIZE, dim=EMBED_DIM):
    n_clusters = random.choice([2, 3, 4])
    centers = np.random.normal(size=(n_clusters, dim))

    rows = []
    for idx in range(n_options):
        cluster = np.random.randint(0, n_clusters)
        emb = centers[cluster] + np.random.normal(scale=np.random.uniform(0.15, 0.9), size=dim)

        quality = np.random.normal(loc=np.random.uniform(0.2, 1.0), scale=0.5)
        usefulness = np.random.normal(loc=quality, scale=0.35)
        utility = 0.65 * quality + 0.35 * usefulness

        rows.append({
            "problem_id": problem_id,
            "option_id": idx,
            "cluster": int(cluster),
            "text": f"Problem {problem_id} - candidate {idx}",
            "utility": float(utility),
            "embedding": emb.astype(float),
        })

    return rows

problems = []
for pid in range(N_PROBLEMS):
    problems.extend(make_problem(pid))

len(problems), problems[0].keys()


## Selection strategies

All strategies must return the same number of final options.


In [ ]:
def select_topk(options, k=FINAL_SET_SIZE):
    ranked = sorted(options, key=lambda x: x["utility"], reverse=True)
    return ranked[:k]

def select_random(options, k=FINAL_SET_SIZE):
    return random.sample(options, k)

def select_diverse(options, k=FINAL_SET_SIZE):
    # Greedy farthest-point selection, seeded by best utility
    remaining = options[:]
    first = max(remaining, key=lambda x: x["utility"])
    selected = [first]
    remaining.remove(first)

    while len(selected) < k and remaining:
        def min_distance_to_selected(candidate):
            c = candidate["embedding"].reshape(1, -1)
            sims = [cosine_similarity(c, s["embedding"].reshape(1, -1))[0, 0] for s in selected]
            return min(sims)

        next_item = min(remaining, key=min_distance_to_selected)
        selected.append(next_item)
        remaining.remove(next_item)
    return selected

def select_cci_aware(options, k=FINAL_SET_SIZE, lam=0.55):
    # Greedy maximize utility while penalizing redundancy with already selected items
    ranked = sorted(options, key=lambda x: x["utility"], reverse=True)
    selected = []

    while len(selected) < k and ranked:
        best_item = None
        best_score = -1e9
        for cand in ranked:
            if not selected:
                score = cand["utility"]
            else:
                sims = [
                    (cosine_similarity(
                        cand["embedding"].reshape(1, -1),
                        s["embedding"].reshape(1, -1)
                    )[0, 0] + 1) / 2
                    for s in selected
                ]
                redundancy_penalty = np.mean(sims)
                score = cand["utility"] - lam * redundancy_penalty
            if score > best_score:
                best_score = score
                best_item = cand

        selected.append(best_item)
        ranked.remove(best_item)
    return selected


## Run the comparison


In [ ]:
all_rows = []
strategy_fns = {
    "confidence_topk": select_topk,
    "random": select_random,
    "diversity": select_diverse,
    "cci_aware": select_cci_aware,
}

for pid in range(N_PROBLEMS):
    opts = [row for row in problems if row["problem_id"] == pid]

    full_utilities = [o["utility"] for o in opts]
    full_embeddings = np.vstack([o["embedding"] for o in opts])
    full_cci = compute_cci(full_utilities, full_embeddings)

    best_utility = max(full_utilities)

    for name, fn in strategy_fns.items():
        selected = fn(opts, FINAL_SET_SIZE)
        sel_utilities = [o["utility"] for o in selected]
        sel_embeddings = np.vstack([o["embedding"] for o in selected])
        sel_cci = compute_cci(sel_utilities, sel_embeddings)

        coverage = int(any(o["utility"] == best_utility for o in selected))
        avg_utility = float(np.mean(sel_utilities))
        regret = float(best_utility - max(sel_utilities))
        gap = float(utility_distribution_gap(sel_utilities))

        all_rows.append({
            "problem_id": pid,
            "strategy": name,
            "full_cci": full_cci["cci"],
            "selected_cci": sel_cci["cci"],
            "delta_cci": sel_cci["cci"] - full_cci["cci"],
            "selected_avg_utility": avg_utility,
            "best_option_covered": coverage,
            "regret": regret,
            "top_gap": gap,
            "selected_entropy": sel_cci["entropy_term"],
            "selected_redundancy": sel_cci["redundancy_term"],
            "selected_ambiguity": sel_cci["ambiguity_term"],
        })

results = pd.DataFrame(all_rows)
results.head()


## Aggregate results


In [ ]:
summary = (
    results.groupby("strategy")[
        [
            "selected_cci",
            "delta_cci",
            "selected_avg_utility",
            "best_option_covered",
            "regret",
            "selected_redundancy",
            "selected_ambiguity",
        ]
    ]
    .mean()
    .sort_values("selected_cci")
)

summary


### Interpretation guide

For this demo, the most important columns are:

- `selected_cci`: lower is better
- `delta_cci`: more negative means larger complexity reduction from the original set
- `selected_avg_utility`: higher is better
- `best_option_covered`: higher is better
- `regret`: lower is better

Because all strategies return the same final number of options, differences are not explained by simply showing fewer items.


## Visual comparison


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

summary["selected_cci"].plot(kind="bar", ax=axes[0], title="Final CCI (lower is better)")
axes[0].set_ylabel("CCI")
axes[0].tick_params(axis='x', rotation=30)

summary["selected_avg_utility"].plot(kind="bar", ax=axes[1], title="Average utility (higher is better)")
axes[1].set_ylabel("Avg utility")
axes[1].tick_params(axis='x', rotation=30)

summary["best_option_covered"].plot(kind="bar", ax=axes[2], title="Best-option coverage (higher is better)")
axes[2].set_ylabel("Coverage rate")
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


## Example problem inspection

This section lets us inspect one option set before and after selection. This is often more useful than aggregate tables because it shows *why* a strategy reduces complexity.


In [ ]:
EXAMPLE_PROBLEM_ID = 3

example = [row for row in problems if row["problem_id"] == EXAMPLE_PROBLEM_ID]
example_df = pd.DataFrame(
    {
        "option_id": [o["option_id"] for o in example],
        "cluster": [o["cluster"] for o in example],
        "utility": [round(o["utility"], 3) for o in example],
        "text": [o["text"] for o in example],
    }
).sort_values("utility", ascending=False)

display(Markdown("### Full candidate set"))
display(example_df)

for name, fn in strategy_fns.items():
    selected = fn(example, FINAL_SET_SIZE)
    selected_df = pd.DataFrame(
        {
            "option_id": [o["option_id"] for o in selected],
            "cluster": [o["cluster"] for o in selected],
            "utility": [round(o["utility"], 3) for o in selected],
            "text": [o["text"] for o in selected],
        }
    ).sort_values("utility", ascending=False)

    sel_cci = compute_cci(
        [o["utility"] for o in selected],
        np.vstack([o["embedding"] for o in selected])
    )

    display(Markdown(f"### Strategy: `{name}`"))
    display(selected_df)
    display(Markdown(
        f"- Final CCI: `{sel_cci['cci']:.3f}`  \
- Redundancy term: `{sel_cci['redundancy_term']:.3f}`  \
- Ambiguity term: `{sel_cci['ambiguity_term']:.3f}`"
    ))


## What this demo supports — and what it does not

This notebook supports a narrow first-paper story:

- complexity can be measured at the set level,
- different fixed-size selectors produce meaningfully different final sets,
- and a complexity-aware selector can reduce redundancy and ambiguity without relying on a smaller final set.

This notebook does **not** establish:

- a final validated cognitive theory,
- a finished benchmark result,
- or a complete multi-domain evaluation.

Those belong to the full paper pipeline, not to this demo notebook.


## Suggested next step after this demo

The next research step is to replace the synthetic utility scaffold with the real benchmark pipeline from the repo and evaluate the same fixed-size comparison design on AmbigNQ or ASQA.
